In [1]:
!nvidia-smi

Tue Apr 11 10:05:04 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.85.12    Driver Version: 525.85.12    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   72C    P8    11W /  70W |      0MiB / 15360MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
!pip install transformers datasets sacremoses

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.7/468.7 kB 29.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.6/880.6 kB 52.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 69.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.1/200.1 kB 20.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.2/212.2 kB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 

In [3]:
from transformers import AutoTokenizer, pipeline
from datasets import load_dataset
import numpy as np

In [4]:
from transformers import HerbertTokenizer, RobertaModel, TFRobertaModel

tokenizer = HerbertTokenizer.from_pretrained("allegro/herbert-klej-cased-tokenizer-v1")
model = TFRobertaModel.from_pretrained("allegro/herbert-klej-cased-v1", from_pt=True)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLMTokenizer'. 
The class this function is called from is 'HerbertTokenizer'.


All PyTorch model weights were used when initializing TFRobertaModel.

All the weights of TFRobertaModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


In [5]:
encoded_input = tokenizer.encode("My name is jamil", "Hello World",return_tensors="tf")
outputs = model(encoded_input)

In [6]:
encoded_input.get_shape()

TensorShape([1, 12])

In [7]:
outputs['pooler_output'].shape

TensorShape([1, 768])

In [8]:
encoded_input

<tf.Tensor: shape=(1, 12), dtype=int32, numpy=
array([[    0,  3397,   125,  2692,  1398,   773, 14484,     1,  7664,
         2591,  5375,     1]], dtype=int32)>

In [9]:
help(tokenizer)

Help on HerbertTokenizer in module transformers.models.herbert.tokenization_herbert object:

class HerbertTokenizer(transformers.tokenization_utils.PreTrainedTokenizer)
 |  HerbertTokenizer(vocab_file, merges_file, tokenizer_file=None, cls_token='<s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>', sep_token='</s>', bos_token='<s>', do_lowercase_and_remove_accent=False, additional_special_tokens=['<special0>', '<special1>', '<special2>', '<special3>', '<special4>', '<special5>', '<special6>', '<special7>', '<special8>', '<special9>'], lang2id=None, id2lang=None, **kwargs)
 |  
 |  Construct a BPE tokenizer for HerBERT.
 |  
 |  Peculiarities:
 |  
 |  - uses BERT's pre-tokenizer: BaseTokenizer splits tokens on spaces, and also on punctuation. Each occurrence of a
 |    punctuation character will be treated separately.
 |  
 |  - Such pretokenized input is BPE subtokenized
 |  
 |  This tokenizer inherits from [`XLMTokenizer`] which contains most of the methods. Users should

In [10]:
model.summary()

Model: "tf_roberta_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 roberta (TFRobertaMainLayer  multiple                 124872192 
 )                                                               
                                                                 
Total params: 124,872,192
Trainable params: 124,872,192
Non-trainable params: 0
_________________________________________________________________


In [11]:
import tensorflow as tf

In [75]:
model.trainable = False
inputs = tf.keras.layers.Input(shape=(None,), dtype=tf.int64, name="tokenized_input")

encoded = model(inputs)['pooler_output']
# hidden_layer = tf.keras.layers.Dense(256, activation="relu", name="hidden_1")(encoded)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="outputs")(encoded)

In [76]:
new_model = tf.keras.Model(inputs=inputs, outputs=outputs)
new_model.summary()

Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 tokenized_input (InputLayer  [(None, None)]           0         
 )                                                               
                                                                 
 tf_roberta_model (TFRoberta  TFBaseModelOutputWithPoo  124872192
 Model)                      lingAndCrossAttentions(l            
                             ast_hidden_state=(None,             
                             None, 768),                         
                              pooler_output=(None, 76            
                             8),                                 
                              past_key_values=None, h            
                             idden_states=None, atten            
                             tions=None, cross_attent            
                             ions=None)                    

In [77]:
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name='acuuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]
new_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

In [78]:
dataset = load_dataset('rotten_tomatoes')
train_dataset = dataset.get('train')
test_dataset = dataset.get('test')
validation_dataset = dataset.get('validation')

  0%|          | 0/3 [00:00<?, ?it/s]

In [79]:
train_df = train_dataset.to_pandas()
validation_df = validation_dataset.to_pandas()

In [80]:
X_train, y_train =  tokenizer(train_df['text'].to_list(), max_length=512, padding=True)['input_ids'], [[y] for y in train_df['label'].to_list()]
X_validation, y_validation =  tokenizer(validation_df['text'].to_list(), max_length=512, padding=True)['input_ids'], [[y] for y in validation_df['label'].to_list()]

/usr/local/lib/python3.9/dist-packages/transformers/tokenization_utils_base.py:2364: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [81]:
X_train, y_train = np.array(X_train).astype(np.int64), np.array(y_train)
X_validation, y_validation = np.array(X_validation).astype(np.int64), np.array(y_validation)


In [82]:
history =  new_model.fit(X_train, y_train, shuffle=True, epochs=100, batch_size = 540, validation_data=[X_validation, y_validation])

Epoch 1/100
16/16 [==============================] - 86s 4s/step - loss: 0.7171 - acuuracy: 0.4995 - precision: 0.4995 - recall: 0.4288 - val_loss: 0.6947 - val_acuuracy: 0.5019 - val_precision: 0.5015 - val_recall: 0.6191
Epoch 2/100
16/16 [==============================] - 63s 4s/step - loss: 0.7026 - acuuracy: 0.4933 - precision: 0.4937 - recall: 0.5217 - val_loss: 0.6933 - val_acuuracy: 0.5197 - val_precision: 0.5212 - val_recall: 0.4841
Epoch 3/100
16/16 [==============================] - 64s 4s/step - loss: 0.6967 - acuuracy: 0.5123 - precision: 0.5130 - recall: 0.4875 - val_loss: 0.6909 - val_acuuracy: 0.5066 - val_precision: 0.5042 - val_recall: 0.7974
Epoch 4/100
16/16 [==============================] - 64s 4s/step - loss: 0.6946 - acuuracy: 0.5151 - precision: 0.5146 - recall: 0.5327 - val_loss: 0.6887 - val_acuuracy: 0.5272 - val_precision: 0.5210 - val_recall: 0.6735
Epoch 5/100
16/16 [==============================] - 59s 4s/step - loss: 0.6937 - acuuracy: 0.5290 - precisi

KeyboardInterrupt: ignored

In [51]:
test_df = test_dataset.to_pandas()
X_test, y_test =  tokenizer(test_df['text'].to_list(), max_length=512, padding=True)['input_ids'], [[y] for y in test_df['label'].to_list()]
X_test, y_test = np.array(X_train).astype(np.int64), np.array(y_train)

/usr/local/lib/python3.9/dist-packages/transformers/tokenization_utils_base.py:2364: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [52]:
y_predicted = new_model.predict(X_validation, batch_size=512)

y_predicted

3/3 [==============================] - 9s 2s/step


array([[0.8478264 ],
       [0.5389999 ],
       [0.6870846 ],
       ...,
       [0.5582545 ],
       [0.6808032 ],
       [0.47296536]], dtype=float32)

In [58]:
X_train.shape

(8530, 96)

array([[20.038664 ],
       [ 4.4601355],
       [19.663084 ],
       ...,
       [ 6.7704086],
       [12.617977 ],
       [ 5.223968 ]], dtype=float32)

In [54]:
X_train_new = model.predict(X_train, batch_size=540)

16/16 [==============================] - 51s 3s/step


In [65]:
X_train_new['pooler_output'].shape

(8530, 768)

In [62]:
inputs = tf.keras.layers.Input(shape=(None,768), dtype=tf.int64, name="tokenized_input")
hidden_layer = tf.keras.layers.Dense(256, activation="relu", name="hidden_1")(inputs)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="outputs")(hidden_layer)
new_model_2 = tf.keras.Model(inputs=inputs, outputs=outputs, name="new_model_2")
new_model_2.summary()

Model: "new_model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 tokenized_input (InputLayer  [(None, None, 768)]      0         
 )                                                               
                                                                 
 hidden_1 (Dense)            (None, None, 256)         196864    
                                                                 
 outputs (Dense)             (None, None, 1)           257       
                                                                 
Total params: 197,121
Trainable params: 197,121
Non-trainable params: 0
_________________________________________________________________


In [63]:
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name='acuuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]
new_model_2.compile(optimizer='adam', loss='binary_crossentropy', metrics=METRICS)

In [67]:
history =  new_model_2.fit(X_train_new['pooler_output'], y_train, shuffle=True, epochs=1000, batch_size = 540)

Epoch 1/1000
16/16 [==============================] - 0s 7ms/step - loss: 0.6932 - acuuracy: 0.4955 - precision: 0.4927 - recall: 0.2992    
Epoch 2/1000
16/16 [==============================] - 0s 6ms/step - loss: 0.6932 - acuuracy: 0.4913 - precision: 0.4924 - recall: 0.5611
Epoch 3/1000
16/16 [==============================] - 0s 6ms/step - loss: 0.6932 - acuuracy: 0.4960 - precision: 0.4954 - recall: 0.4263
Epoch 4/1000
16/16 [==============================] - 0s 6ms/step - loss: 0.6932 - acuuracy: 0.4930 - precision: 0.4931 - recall: 0.4994
Epoch 5/1000
16/16 [==============================] - 0s 6ms/step - loss: 0.6931 - acuuracy: 0.5000 - precision: 0.0000e+00 - recall: 0.0000e+00
Epoch 6/1000
16/16 [==============================] - 0s 6ms/step - loss: 0.6931 - acuuracy: 0.4986 - precision: 0.4991 - recall: 0.7454
Epoch 7/1000
16/16 [==============================] - 0s 6ms/step - loss: 0.6932 - acuuracy: 0.5000 - precision: 0.5000 - recall: 1.0000
Epoch 8/1000
16/16 [=========

KeyboardInterrupt: ignored